In [6]:
from typing import Dict, List, Any
import json
from pathlib import Path
import matplotlib; matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from libpysal.weights import lag_spatial
from s_quantum_utils import execute_code
import numpy as np
import tqdm as tqdm_mod

# ── 路径配置(本地)─────────────────────────────────────────────────────
REPO_DIR   = Path("/Users/milly/Desktop/LLM-based-Q-generation")
EXP_DIR    = REPO_DIR / "dataset"
QHE_CKPT = EXP_DIR / "starcoder_qhe_results_t02.json"

# ── 加载 S 分数(已由RQ2_curpus_similarity.ipynb 算好)────────────────────
CORPUS_DIR = Path("/Users/milly/Desktop/LLM-based-Q-generation/exp1_results/similar_score")

qhe_records = json.load(open(QHE_CKPT))
print(f"QHE loaded: {len(qhe_records)} records" )
import sys as _sys; _sys.path.insert(0, str(REPO_DIR))

QHE loaded: 151 records


In [ ]:
class QHGEvaluator:
    """Qiskit HumanEval评测器"""

    def __init__(self, timeout: int = 30):
        self.timeout = timeout

    def combine_code(self, prompt: str, generated: str, entry_point: str = None) -> str:
        """
        核心功能1：代码合并(目的：对不同模型产生的代码做兼容)
        合并prompt和生成的代码，把模型生成的代码和原始prompt正确拼装
        为了保证客观执行本次不使用，改用qiskit_humaneval.py的combine_code方法
        """
        if entry_point and re.search(rf"def\s+{re.escape(entry_point)}\s*\(", generated):
            return generated

        if not self._looks_like_python_code(generated):
            # 如果不是Python代码，尝试从中提取代码块
            generated = self._extract_code_from_text(generated)

        # 1.获取缩进级别
        indent = self._get_function_body_indent(prompt)
        # 2.规范化缩进
        normalized = self._normalize_indentation(generated, indent)
        return prompt + "\n" + normalized

    def _looks_like_python_code(self, text: str) -> bool:
        """判断文本看起来像Python代码"""
        # 检查是否有Python关键字
        python_keywords = ['def ', 'return ', 'import ', 'from ', 'class ', 'if ', 'for ', 'while ']
        for keyword in python_keywords:
            if keyword in text:
                return True
        # 检查是否有常见语法结构
        if re.search(r'[a-zA-Z_][a-zA-Z0-9_]*\s*\(', text):  # 函数调用
            return True
        if re.search(r'=\s*[a-zA-Z_][a-zA-Z0-9_]*\(', text):  # 赋值+函数调用
            return True
        return False

    def _extract_code_from_text(self, text: str) -> str:
        """从文本中提取代码块"""
        # 查找 ```python 或 ``` 代码块
        code_block_pattern = r'```(?:python)?\s*\n(.*?)```'
        code_blocks = re.findall(code_block_pattern, text, re.DOTALL)
        if code_blocks:
            return code_blocks[0].strip()

        # 如果没有代码块，尝试提取看起来像代码的行
        lines = text.split('\n')
        code_lines = []

        for line in lines:
          stripped = line.strip()
          # 只保留真正的代码行
          if (stripped.startswith('def ') or
              stripped.startswith('return ') or
              stripped.startswith('from ') or
              stripped.startswith('import ') or
              re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*\s*=', stripped) or  # 赋值语句
              re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*\(', stripped)):     # 函数调用
              code_lines.append(line)
          # 遇到空行时，如果已经收集了代码，停止
          elif code_lines and not stripped:
              break

        if code_lines:
            return '\n'.join(code_lines)

        return text

    def _get_function_body_indent(self, prompt: str) -> int:
        """获取函数体的预期缩进
        找到prompt中最后一个def行，计算该行前面的空格数，然后+4得到函数体应有的缩进。如果没找到，默认缩进4空格。
        """
        lines = prompt.split("\n")
        for line in reversed(lines):
            stripped = line.lstrip()
            if stripped.startswith('def '):
                return len(line) - len(stripped) + 4
        return 4

    def _normalize_indentation(self, code: str, target_indent: int) -> str:
        """规范化缩进
        原理：
            找出第一行非空代码的缩进作为基准
            计算每一行相对于基准的缩进量
            在目标缩进基础上加上相对缩进量，得到新缩进
            这样保证代码的结构关系不变，只是整体偏移到正确位置
        例子：
            text
            模型生成：
            circuit = QuantumCircuit(2)
                circuit.h(0)
                circuit.cx(0,1)

            目标缩进4空格后：
                circuit = QuantumCircuit(2)
                    circuit.h(0)
                    circuit.cx(0,1)
        """

        lines = code.split("\n")
        if not lines:
            return code

        # 找到第一行**非空且非注释**的行的缩进作为基准
        first_code_line = None
        first_code_indent = 0

        for line in lines:
            stripped = line.strip()
            if stripped and not stripped.startswith('#'):
                first_code_line = line
                first_code_indent = len(line) - len(line.lstrip())
                break

        # 如果没有找到代码行(全是注释)，用第一行作为基准
        if first_code_line is None:
            first_line = next((line for line in lines if line.strip()), "")
            if not first_line:
                return code
            first_code_indent = len(first_line) - len(first_line.lstrip())

        # 重新缩进
        result = []
        for line in lines:
            if not line.strip():
                result.append("")
                continue

            stripped = line.lstrip()
            current_indent = len(line) - len(stripped)

            # 如果是注释行，特殊处理
            if stripped.startswith('#'):
                # 注释行保持原样，不加额外缩进
                result.append(line)
            else:
                # 代码行：计算相对缩进
                relative = current_indent - first_code_indent
                new_indent = max(0, target_indent + relative)
                result.append(" " * new_indent + stripped)

        return "\n".join(result)

    def execute_code(self, code: str, test_code: str, entry_point: str) -> Dict[str, Any]:
        """QHE简化版"""
        # 组装完整测试代码
        full_code = f"{code}\n\n{test_code}"
        try:
          import builtins as _builtins
          _BLOCKED_BUILTINS = {
                # "open",         # filesystem read/write // needed by qiskitHumanEval/82
                "exec",         # nested arbitrary code execution
                "eval",         # expression-level arbitrary execution
                "compile",      # bytecode compilation (precursor to exec/eval)
                "breakpoint",   # drops into debugger / pdb shell
                "input",        # blocks on stdin in CI; not needed by solutions
            }
          restricted_builtins = {
                k: v
                for k, v in vars(_builtins).items()
                if k not in _BLOCKED_BUILTINS
            }
          namespace: dict[str, Any] = {"__builtins__": restricted_builtins}
          print(f"[DEBUG] 执行的代码:\n{full_code}")
          exec(full_code, namespace)
          namespace["check"](namespace[entry_point])

            # Run the check function against the implemented function
          return {"success": True, "error": "", "output": ""}
        except TimeoutError as e:
            return {"success": False, "error": "Timeout", "output": "TimeoutError"}
        except AssertionError as e:
          #print(f'报错AssertionError：{str(e)}')
          return {"success": False, "error": str(e), "output": "AssertionError"}
        except Exception as e:
          #print(f'报错Exception：{str(e)}')
          return {"success": False, "error": str(e), "output": "Exception"}



In [121]:
sys.path.insert(0, str( REPO_DIR/ "Quanbench"))
from Quanbench_eval.execution import check_correctness

evaluator    = QHGEvaluator(timeout=60)

# ═══════════════════════════════════════════════════════════════
# 关键
# Step 1 — Execute (run ONCE per model, results saved to disk)
#   exec_results: [bool, ...]  one bool per code in raw_codes
# ═══════════════════════════════════════════════════════════════

def execute_qhe(records: list, evaluator, out_json: Path) -> list:
    """Execute QHE tasks via QHGEvaluator.execute_code. Resume-safe."""
    done = {}
    if out_json.exists():
        for r in json.load(open(out_json)):
            if "exec_results" in r:
                done[r["task_id"]] = r
    todo = [r for r in records if r["task_id"] not in done]
    print(f"QHE execute: {len(done)} cached, {len(todo)} to run")
    for r in tqdm_mod.tqdm(todo):
        codes = r.get("raw_codes", [])
        exec_results = [
            execute_code(
                r["prompt"] + "\n    " + code,
                r.get("test", ""),
                r.get("entry_point", ""),
                timeout=10.0,
            )["success"]
            for code in codes
        ]
        done[r["task_id"]] = {**r, "exec_results": exec_results}
        json.dump(list(done.values()), open(out_json, "w", encoding="utf-8"),
                  indent=2, ensure_ascii=False)
    return list(done.values())


def execute_qbench(records: list, problems_dict: dict, out_json: Path,
                   timeout: float = 30.0) -> list:
    """Execute QuanBench tasks using the official task-specific tests."""
    done = {}
    if out_json.exists():
        for r in json.load(open(out_json)):
            if "exec_results" in r:
                done[r["task_id"]] = r
    todo = [r for r in records if r["task_id"] not in done]
    print(f"QB  execute: {len(done)} cached, {len(todo)} to run")
    for r in tqdm_mod.tqdm(todo):
        task_id = r["task_id"]
        problem = problems_dict.get(task_id)
        prompt = r.get("prompt", "")
        if not problem:
            continue
        exec_results = [
            check_correctness(problem, prompt + "\n    " + code, timeout, completion_id=i)["passed"]
            for i, code in enumerate(r.get("raw_codes", []))
        ]
        done[task_id] = {**r, "exec_results": exec_results}
        json.dump(list(done.values()), open(out_json, "w", encoding="utf-8"),
                  indent=2, ensure_ascii=False)
    return list(done.values())


# ═══════════════════════════════════════════════════════════════
# 
# Step 2 — pass@k from stored exec_results (NO re-execution)
#   decontamination = filter records, then call these functions
# ═══════════════════════════════════════════════════════════════

def _estimator(n: int, c: int, k: int) -> float:
    """Unbiased HumanEval pass@k for one task. Returns 0-1."""
    if n - c < k:
        return 1.0
    return 1.0 - float(np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))

def pass_at_k_from_exec(exec_records: list, k: int = 1,
                         n_samples= None) -> float:
    """Benchmark pass@k from stored exec_results. Returns percentage 0-100.
    n_samples: if set, truncate exec_results to first n_samples (e.g. 5 to mimic old run).
    """
    vals = []
    for r in exec_records:
        er = r.get("exec_results", [])
        if n_samples is not None:
            er = er[:n_samples]
        if not er:
            continue
        n, c = len(er), sum(1 for x in er if x)
        if n < k:
            continue
        vals.append(_estimator(n, c, k))
    return float(np.mean(vals)) * 100 if vals else float("nan")


def pass_at_k_de_from_exec(exec_records: list, exclude_ids: set,
                            k: int = 1,
                            n_samples= None) -> float:
    """Decontaminated pass@k — filter then compute, no re-execution."""
    subset = [r for r in exec_records if r["task_id"] not in exclude_ids]
    return pass_at_k_from_exec(subset, k, n_samples=n_samples)


problems_dict loaded: 44 tasks


In [111]:
# -----------执行benchmark — QHE ----------------
# execute_qhe is resume-safe: loads existing cache, only runs missing tasks.

QHE_EXEC_CACHE = EXP_DIR / "starcoder_qhe_exec.json"
qhe_exec = execute_qhe(qhe_records, evaluator, QHE_EXEC_CACHE)
print(f"Done — QHE: {len(qhe_exec)} tasks")

QHE execute: 0 cached, 151 to run


  1%|▏         | 2/151 [00:00<00:11, 13.33it/s]/Users/milly/Library/Python/3.9/lib/python/site-packages/qiskit/visualization/circuit/matplotlib.py:287: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  mpl_figure = plt.figure()
 10%|▉         | 15/151 [00:01<00:09, 13.76it/s]<string>:9: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
<string>:9: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
<string>:9: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is 

{'111': 512, '000': 512}


/Users/milly/Library/Python/3.9/lib/python/site-packages/qiskit/circuit/quantumcircuit.py:4625: UserWarning: Trying to add QuantumRegister to a QuantumCircuit having a layout
  circ.add_register(qreg)
 24%|██▍       | 36/151 [00:03<00:08, 13.87it/s]

┌───┐ ░                      ░ ┌───┐┌─┐         
q_0: ┤ H ├─░───■──────────────────░─┤ H ├┤M├─────────
     ├───┤ ░   │                  ░ ├───┤└╥┘┌─┐      
q_1: ┤ H ├─░───┼────■─────────────░─┤ H ├─╫─┤M├──────
     ├───┤ ░   │    │             ░ ├───┤ ║ └╥┘┌─┐   
q_2: ┤ H ├─░───┼────┼────■────────░─┤ H ├─╫──╫─┤M├───
     ├───┤ ░   │    │    │        ░ ├───┤ ║  ║ └╥┘┌─┐
q_3: ┤ H ├─░───┼────┼────┼────■───░─┤ H ├─╫──╫──╫─┤M├
     └───┘ ░ ┌─┴─┐┌─┴─┐┌─┴─┐┌─┴─┐ ░ └───┘ ║  ║  ║ └╥┘
q_4: ──────░─┤ X ├┤ X ├┤ X ├┤ X ├─░───────╫──╫──╫──╫─
           ░ └───┘└───┘└───┘└───┘ ░       ║  ║  ║  ║ 
c: 4/═════════════════════════════════════╩══╩══╩══╩═
                                          0  1  2  3

...

{'01': 237, '10': 253, '11': 282, '00': 252}
{'01': 221, '00': 240, '11': 283, '10': 280}


 62%|██████▏   | 93/151 [00:06<00:02, 20.42it/s]

StabilizerState(['+XX', '+ZZ'])
[0.5 0.  0.  0.5]


100%|██████████| 151/151 [01:16<00:00,  1.96it/s]

Done — QHE: 151 tasks


In [123]:
# ── 计算全量 pass@k(从已存exec结果计算，不执行)────────────────────────────
k = 1
n = 20
p_full_qhe    = pass_at_k_from_exec(qhe_exec, k) # n=20
print("=== QHE pass@k ===")
print(f"  n=20  pass@{k} = {p_full_qhe:.2f}%   ({len(qhe_exec)} tasks)")


=== QHE pass@k ===
  n=20  pass@1 = 21.72%   (151 tasks)


In [122]:
print("=== QHE pass@k n=5  ===")
p_full_qhe_n5 = pass_at_k_from_exec(qhe_exec, k, n_samples=5)      # n=5 (前5个)
print(f"  n=5   pass@{k} = {p_full_qhe_n5:.2f}%   ({len(qhe_exec)} tasks, 前5个sample)")


=== QHE pass@k n=5  ===
  n=5   pass@1 = 21.85%   (151 tasks, 前5个sample)


13.计算去污染之后的Drop

In [ ]:


def _load_s_top1(path: Path) -> dict:
    """读 stack_scores.json → {task_id: max(surface_top1, semantic_top1)}"""
    raw = json.load(open(path))
    return {tid: max(v["surface_top1"], v["semantic_top1"]) for tid, v in raw.items()}


def _print_s_dist(name: str, s_dict: dict, step: int = 20) -> None:
    buckets = {}
    for s in s_dict.values():
        b = int(s // step) * step
        buckets[b] = buckets.get(b, 0) + 1
    print(f"\n{name} S score distribution (bucket width={step}):")
    for b in sorted(buckets):
        bar = "█" * buckets[b]
        print(f"  [{b:3d},{b+step:3d})  {buckets[b]:3d}  {bar}")



QHE-baseline score被错误覆盖了，后面重跑后记得改
#现在用的是修复的数据，QHE从compare里取，QB的5和38用中位数填充

In [114]:
qhe_S_top1 = _load_s_top1(CORPUS_DIR / "qhe_recovery_scores.json")
print(f"QHE S scores: {len(qhe_S_top1)} tasks")
_print_s_dist("QHE", qhe_S_top1)

QHE S scores: 151 tasks

QHE S score distribution (bucket width=20):
  [ 40, 60)   72  ████████████████████████████████████████████████████████████████████████
  [ 60, 80)   67  ███████████████████████████████████████████████████████████████████
  [ 80,100)   12  ████████████


去污染逻辑

In [115]:

THRESHOLDS = [60,70,80]  # [60, 65, 70, 80]
print("QHE 去污染 pass@1(%)===")
print(f" pass@{k}_o = {p_full_qhe:.2f}")
for thr in THRESHOLDS:
    excl = {r["task_id"] for r in qhe_exec if qhe_S_top1.get(r["task_id"], 0) >= thr}
    p_dc = pass_at_k_de_from_exec(qhe_exec, exclude_ids=excl, k=1)
    drop =  p_dc - p_full_qhe
    rm = len(excl)/len(qhe_exec)
    print(f"     S>{thr} = {p_dc:.2f}   DROP={drop:7.2f}({drop/p_full_qhe*100:+7.2f}%) Rm ={ rm:.2f} ")



QHE 去污染 pass@1(%)===
 pass@1_o = 21.72
     S>60 = 14.58   DROP=  -7.14( -32.86%) Rm =0.52 
     S>70 = 20.66   DROP=  -1.06(  -4.87%) Rm =0.25 
     S>80 = 22.23   DROP=   0.51(  +2.34%) Rm =0.08 


In [124]:
# ── n=5 (前5个 sample，模拟旧结果) ────────────────────────────
print("QHE 去污染 pass@1(%) [n=5]  ===")
print(f" pass@{k}_o = {p_full_qhe_n5:.2f}")
for thr in THRESHOLDS:
    excl  = {r["task_id"] for r in qhe_exec if qhe_S_top1.get(r["task_id"], 0) >= thr}
    p_dc  = pass_at_k_de_from_exec(qhe_exec, exclude_ids=excl, k=1, n_samples=5)
    drop  = p_dc - p_full_qhe_n5
    rm    = len(excl) / len(qhe_exec)
    print(f"     S>{thr} = {p_dc:.2f}   DROP={drop:7.2f}({drop/p_full_qhe_n5*100:+7.2f}%) Rm={rm:.2f}")

QHE 去污染 pass@1(%) [n=5]  ===
 pass@1_o = 21.85
     S>60 = 13.61   DROP=  -8.24( -37.72%) Rm=0.52
     S>70 = 20.88   DROP=  -0.97(  -4.44%) Rm=0.25
     S>80 = 22.16   DROP=   0.30(  +1.39%) Rm=0.08


In [29]:
# ── Section 14: Difficulty-Controlled DROP ─────────────────────────────────
# QHE: 用官方 difficulty_scale (basic/intermediate/difficult)
# QB : 无官方难度，按算法复杂度手动分类(对齐 QHE 三档)
# DROP_bracket = p_de - p_full  (负值=seen任务更好=污染信号)

import json as _json

# ── QHE difficulty_scale ─────────────────────────────────────────────────────
_qhe_raw = _json.load(open(
    REPO_DIR / 'qiskit-human-eval/dataset/dataset_qiskit_test_human_eval.json'
))
_DIFF_ORDER = {'basic': 0, 'intermediate': 1, 'difficult': 2}
qhe_difficulty = {
    t['task_id']: t['difficulty_scale'] for t in _qhe_raw if 'difficulty_scale' in t
}

# ── QB 手动难度映射(按算法复杂度，对齐QHE三档)────────────────────────────
# basic:        标准教科书电路(Bell, GHZ, BV, DJ, Qadd 等)
# intermediate: 标准量子算法(QFT, QPE, Simon, Grover, 门分解 等)
# difficult:    研究级/参数化电路(HHL, QAOA, Shor, VQE, SWAP-test 等)
QB_DIFFICULTY = {
    '16':'basic',  '17':'basic',  '21':'basic',  '22':'basic',  '23':'basic',
    '24':'basic',  '29':'basic',  '30':'basic',  '35':'basic',  '11':'basic',
    '12':'basic',  '20':'basic',  '36':'basic',
    '01':'intermediate', '08':'intermediate', '09':'intermediate',
    '13':'intermediate', '14':'intermediate', '15':'intermediate',
    '18':'intermediate', '19':'intermediate', '02':'intermediate',
    '25':'intermediate', '26':'intermediate', '27':'intermediate',
    '28':'intermediate', '33':'intermediate', '34':'intermediate',
    '37':'intermediate', '42':'intermediate', '43':'intermediate',
    '44':'intermediate', '31':'intermediate',
    '03':'difficult', '04':'difficult', '06':'difficult', '07':'difficult',
    '10':'difficult', '32':'difficult', '39':'difficult',
    '40':'difficult', '41':'difficult',
}

print('QHE difficulty:', {k: sum(1 for v in qhe_difficulty.values() if v==k) for k in _DIFF_ORDER})
from collections import Counter as _Cnt
print('QB  difficulty:', dict(_Cnt(QB_DIFFICULTY.values())))


QHE difficulty: {'basic': 79, 'intermediate': 67, 'difficult': 5}
QB  difficulty: {'basic': 13, 'intermediate': 20, 'difficult': 9}


In [126]:
def drop_by_difficulty(
    exec_records: list,
    s_scores: dict,
    difficulty_map: dict,
    score_threshold: float = 80,
    label: str = '',
    k: int = 1,
    n_samples=None,
):
    """
    Difficulty-controlled DROP.
    difficulty_map: {task_id: label}  ('basic'/'intermediate'/'difficult')
    Each label = one bracket; within bracket, S >= score_threshold = 'seen'.
    DROP_bracket = pass@k(unseen) - pass@k(all)   (negative = contamination signal)
    """
    id_to_r = {r['task_id']: r for r in exec_records}
    from collections import defaultdict
    groups = defaultdict(list)
    for tid, r in id_to_r.items():
        if tid in s_scores and r.get('exec_results') and tid in difficulty_map:
            groups[difficulty_map[tid]].append(
                (tid, s_scores[tid])
            )
    ordered = sorted(groups.keys(), key=lambda x: _DIFF_ORDER.get(x, 99))

    print(f'\n{"="*68}')
    print(f'  {label}  Difficulty-bracket DROP  '
          f'(Rm: S>={score_threshold}, k={k})')
    print(f'{"="*68}')
    print(f'  {"bracket":>14}  {"n":>4}  {"n_rm":>6}  '
          f'{"p_full%":>7}  {"p_de%":>7}  {"DROP":>8}')
    print(f'  {"-"*58}')

    bracket_drops = []
    for lbl in ordered:
        bt       = groups[lbl]          # [(tid, S), ...]
        rm_ids = {tid for tid, s in bt if s >= score_threshold}
        de_ids   = {tid for tid, _ in bt if tid not in rm_ids}

        def p_k(ids, _r=id_to_r, _k=k):
            vals = []
            for t in ids:
                er = _r[t]['exec_results']
                if n_samples is not None:
                    er = er[:n_samples]
                if er:
                    vals.append(_estimator(len(er), sum(er), _k))
            return np.mean(vals) * 100 if vals else float('nan')

        p_full = p_k({tid for tid, _ in bt})
        p_de   = p_k(de_ids)
        drop   = p_de - p_full
        bracket_drops.append(drop)

        print(f'  {lbl:>14}  {len(bt):>4d}  {len(rm_ids):>6d}  '
              f'{p_full:>6.2f}  {p_de:>6.2f}  {drop:>+7.2f}({np.nan if p_full == 0 else (drop / p_full) * 100:+7.2f}%)')

    valid  = [d for d in bracket_drops if not np.isnan(d)]
    mean_d = np.mean(valid) if valid else float('nan')
    print(f'  {"-"*58}')
    print(f'  {"(不做参考)bracket 平均 DROP":>48}  {mean_d:>+7.2f}%')
    return bracket_drops


In [116]:
# ── QHE ─────────────────────────────────────────────────────────────────────
qhe_r_bracket_drops = drop_by_difficulty(
    qhe_exec, qhe_S_top1, qhe_difficulty,
    score_threshold=60, label='QHE', k=k,
)


  QHE  Difficulty-bracket DROP  (Rm: S>=60, k=1)
         bracket     n    n_rm  p_full%    p_de%      DROP
  ----------------------------------------------------------
           basic    79      51   26.84   11.96   -14.87( -55.42%)
    intermediate    67      28   17.31   18.33    +1.02(  +5.89%)
       difficult     5       0    0.00    0.00    +0.00(   +nan%)
  ----------------------------------------------------------
                             (不做参考)bracket 平均 DROP    -4.62%


In [127]:
# ── QHE n=5 (前5个 sample) ───────────────────────────────────────────────────
qhe_r_bracket_drops_n5 = drop_by_difficulty(
    qhe_exec, qhe_S_top1, qhe_difficulty,
    score_threshold=60, label='QHE [n=5]', k=k, n_samples=5,
)


  QHE [n=5]  Difficulty-bracket DROP  (Rm: S>=60, k=1)
         bracket     n    n_rm  p_full%    p_de%      DROP
  ----------------------------------------------------------
           basic    79      51   27.09    9.29   -17.80( -65.72%)
    intermediate    67      28   17.31   18.46    +1.15(  +6.63%)
       difficult     5       0    0.00    0.00    +0.00(   +nan%)
  ----------------------------------------------------------
                             (不做参考)bracket 平均 DROP    -5.55%


In [150]:
# -----------执行benchmark — QB ----------------
# execute_qbench is resume-safe: loads existing cache, only runs missing tasks.

QB_CKPT  = EXP_DIR / "starcoder_qb_results_t02.json"
qb_records  = json.load(open(QB_CKPT))
QB_EXEC_CACHE = EXP_DIR / "starcoder_qb_exec.json"

# ── QuanBench problem definitions (needed by execute_qbench) ─────────────────
_qb_jsonl = REPO_DIR / 'Quanbench/Quanbench_eval/problem_sets/quanbench44/Quanbench.jsonl'
with open(_qb_jsonl, encoding='utf-8-sig') as _f:
    problems_dict = {d['task_id']: d for d in (json.loads(l) for l in _f if l.strip())}
print(f'problems_dict loaded: {len(problems_dict)} tasks')

qb_exec = execute_qbench(qb_records, problems_dict, QB_EXEC_CACHE)
print(f"Done — QB: {len(qb_exec)} tasks")

print("\n=== QB pass@1 ===")
p_full_qb = pass_at_k_from_exec(qb_exec, k)
print(f"  Full Pass@k_o:  {p_full_qb:.2f}% , k={k}       ({len(qb_exec)} tasks), n={n}")



problems_dict loaded: 44 tasks
QB  execute: 44 cached, 0 to run


0it [00:00, ?it/s]

Done — QB: 44 tasks

=== QB pass@1 ===
  Full Pass@k_o:  10.45% , k=1       (44 tasks), n=20


In [151]:
qb_S_top1  = _load_s_top1(CORPUS_DIR / "qb_stack_scores.json")
print(f"QB  S scores: {len(qb_S_top1)} tasks")
_print_s_dist("QB",  qb_S_top1)

    
print("\nQB 去污染 pass@1(%) ===")
print(f" pass@{k}_o = {p_full_qb:.2f}")
for thr in THRESHOLDS:
    excl = {r["task_id"] for r in qb_exec if qb_S_top1.get(r["task_id"], 0) >= thr}
    p_dc = pass_at_k_de_from_exec(qb_exec, exclude_ids=excl, k=1)
    drop = p_dc - p_full_qb
    rm = len(excl)/len(qb_exec)
    print(f"     S>{thr} = {p_dc:.2f}  DROP={drop:.2f}({drop/p_full_qb*100:+7.2f}%)  Rm ={ rm:.2f}")


# ── QB ──────────────────────────────────────────────────────────────────────
qb_bracket_drops = drop_by_difficulty(
    qb_exec, qb_S_top1, QB_DIFFICULTY,
    score_threshold=70, label='QB', k=k,
)

QB  S scores: 44 tasks

QB S score distribution (bucket width=20):
  [ 40, 60)   16  ████████████████
  [ 60, 80)   26  ██████████████████████████
  [ 80,100)    2  ██

QB 去污染 pass@1(%) ===
 pass@1_o = 10.45
     S>60 = 5.31  DROP=-5.14( -49.18%)  Rm =0.64
     S>70 = 6.00  DROP=-4.45( -42.61%)  Rm =0.32
     S>80 = 9.17  DROP=-1.29( -12.32%)  Rm =0.05

  QB  Difficulty-bracket DROP  (Rm: S>=70, k=1)
         bracket     n    n_rm  p_full%    p_de%      DROP
  ----------------------------------------------------------
           basic    13       5   20.00   10.63    -9.38( -46.88%)
    intermediate    20       6   10.00    6.79    -3.21( -32.14%)
       difficult     9       3    0.00    0.00    +0.00(   +nan%)
  ----------------------------------------------------------
                             (不做参考)bracket 平均 DROP    -4.20%


R6.对齐Drop波动 最大的rm比例:
QHE：52%，QB：32%

In [152]:
def random_drop_breakdown(exec_records, pct=0.33, k=1, seed=None):
    """随机移除 pct 比例任务，返回 Drop详情。"""
    rng     = np.random.default_rng(seed)
    all_ids = [r['task_id'] for r in exec_records]
    n_rm    = max(1, int(len(all_ids) * pct))
    excl    = set(rng.choice(all_ids, size=n_rm, replace=False).tolist())
    p_full  = pass_at_k_from_exec(exec_records, k)
    p_de    = pass_at_k_de_from_exec(exec_records, excl, k)
    drop_detail = {
        'p_de':p_de,
        'p_full':p_full,
        'p_drop':p_de - p_full
    }
    return drop_detail
# ── 单次 ─────────────────────────────────────────────────────────────────────
qhe_rand_full = random_drop_breakdown(qhe_exec, pct=0.53, k=k, seed=42)['p_full']
qb_rand_full  = random_drop_breakdown(qb_exec,  pct=0.32, k=k, seed=42)['p_full']

# ── 多次收敛(500次取均值±std)───────────────────────────────────────────────
N_ITER = 500
qhe_rand_dist = [random_drop_breakdown(qhe_exec, pct=0.53, k=k, seed=i)['p_drop'] for i in range(N_ITER)]
qb_rand_dist  = [random_drop_breakdown(qb_exec,  pct=0.32, k=k, seed=i)['p_drop'] for i in range(N_ITER)]

qhe_rand_mean, qhe_rand_std = np.mean(qhe_rand_dist), np.std(qhe_rand_dist)
qb_rand_mean,  qb_rand_std  = np.mean(qb_rand_dist),  np.std(qb_rand_dist)


# ── 汇总打印 ──────────────────────────────────────────────────────────────────
print(f'\n{"="*72}')
print(f'  Random Remove vs baseline(%)  ( k={k})')
print(f'{"="*72}')
print(f'  {"":6s}  {"Pass@k{k}_full":>12s}  '
      f'{"Drop_random mean±std (n=500)":>24s}  {"Relative Difference":>13s}')
print(f'  {"-"*62}')
print(f'  {"QHE (Rm:0.52)":6s}  {qhe_rand_full:>.2f}  '
      f'{qhe_rand_mean:>+8.2f} ± {qhe_rand_std:.2f}        '
      f'({qhe_rand_mean/qhe_rand_full*100:7.2f}%)')

print(f'  {"QB (Rm:0.32)":6s}  {qb_rand_full:>.2f}  '
      f'{qb_rand_mean:>+8.2f} ± {qb_rand_std:.2f}        '
      f'({qb_rand_mean/qb_rand_full*100:+7.2f}%)')


  Random Remove vs baseline(%)  ( k=1)
          Pass@k{k}_full  Drop_random mean±std (n=500)  Relative Difference
  --------------------------------------------------------------
  QHE (Rm:0.52)  21.72     -0.10 ± 2.89        (  -0.44%)
  QB (Rm:0.32)  10.45     +0.20 ± 2.13        (  +1.94%)


## Build corpus snippet records

In [97]:
# ══════════════════════════════════════════════════════════════════
#  Section . 跑 QHE（QB有自己的corpus matches文件后，直接把path换掉调同一个函数）
# ══════════════════════════════════════════════════════════════════
import json

with open('exp1_results/similar_score/qhe_stack_matches.json', encoding='utf-8') as f:
    qhe_stack_matches = json.load(f)
with open('exp1_results/similar_score/qb_stack_matches.json', encoding='utf-8') as f:
    qb_stack_matches = json.load(f)

In [147]:
# ══════════════════════════════════════════════════════════════════
#  find_minimal_circuit_snippet  —  execution-side QC slice (corpus_snippet_qc_slice)
#
#  DISTINCT from find_best_surface_snippet (text-side, corpus_snippet):
#    find_best_surface_snippet   : raw-text sliding-window rapidfuzz alignment over the
#                                  corpus file → corpus_snippet (raw source region, used
#                                  for s_gate_ast / s_struct text metrics).
#    find_minimal_circuit_snippet: AST-enumerates every `var = QuantumCircuit(...)` in
#                                  the corpus file, extracts a minimal dependency closure
#                                  per candidate via ast.unparse(), then picks the
#                                  candidate whose reconstructed text has the highest
#                                  surface similarity to the canonical body →
#                                  corpus_snippet_qc_slice (used for circuit execution).
#
#  'snippet_status: ok' means AST found a QC constructor and ast.unparse() succeeded.
#  It does NOT mean the snippet executes — runtime failures are common
#  (missing deps, deprecated Qiskit 0.x API, etc.).
#
#  Known bias: ALL file-level imports are prepended to every candidate (module_imports).
#  When multiple tasks retrieve the same corpus file their qc_slices may be identical.
# ══════════════════════════════════════════════════════════════════
import json as _json_snip
from pathlib import Path as _snip_Path
import ast
def _normalize_for_surface(code: str) -> str:
    """Must remain identical to R2_corpus_similarity.ipynb."""
    import re as _re
    code = _re.sub(r'#.*$', '', code, flags=_re.MULTILINE)
    code = _re.sub(r'""".*?"""', '', code, flags=_re.DOTALL)
    code = _re.sub(r"'''.*?'''", '', code, flags=_re.DOTALL)
    return ' '.join(code.split()).lower()

def _normalize_with_source_map(code: str):
    """Normalize code and retain the source position of each character."""
    import re

    chars = list(code)
    source_map = list(range(len(code)))

    patterns = [
        (r'#.*$', re.MULTILINE),
        (r'""".*?"""', re.DOTALL),
        (r"'''.*?'''", re.DOTALL),
    ]

    for pattern, flags in patterns:
        text = ''.join(chars)
        remove = [False] * len(chars)

        for match in re.finditer(pattern, text, flags):
            for i in range(match.start(), match.end()):
                remove[i] = True

        chars = [
            char for i, char in enumerate(chars)
            if not remove[i]
        ]
        source_map = [
            position for i, position in enumerate(source_map)
            if not remove[i]
        ]

    text = ''.join(chars)
    normalized_chars = []
    normalized_map = []

    tokens = list(re.finditer(r'\S+', text))

    for token_index, token in enumerate(tokens):
        if token_index > 0:
            normalized_chars.append(' ')
            normalized_map.append(source_map[token.start()])

        for i in range(token.start(), token.end()):
            normalized_chars.append(chars[i].lower())
            normalized_map.append(source_map[i])

    return ''.join(normalized_chars), normalized_map


def find_minimal_circuit_snippet(code: str, query_code: str) -> dict:
    """
    AST-based extraction of a QuantumCircuit-building code block from a corpus file.

    Unlike find_best_surface_snippet (raw text alignment → corpus_snippet for text
    metrics), this function:
      1. AST-parses the full corpus file and enumerates every `var = QuantumCircuit(...)`
         assignment together with its minimal statement dependency closure.
      2. Reconstructs each candidate via ast.unparse() — syntactically valid but NOT
         guaranteed to execute (runtime deps, deprecated API, etc. may still fail).
      3. Prepends ALL file-level import statements to every candidate (module_imports).
      4. Selects the candidate with the highest surface similarity to query_code.

    Returns corpus_snippet_qc_slice + target_var for direct exec() → namespace lookup.
    snippet_status 'ok' = AST found a QC constructor; does not imply runtime success.
    """
    from rapidfuzz import fuzz as _fuzz
    import hashlib as _hashlib

    try:
        tree = ast.parse(code)
    except SyntaxError:
        return {'snippet_status': 'corpus_parse_failed', 'candidate_count': 0,
                'corpus_snippet': None, 'target_var': None}

    query_norm = _normalize_for_surface(_strip_preamble_for_surface(query_code))
    file_norm  = _normalize_for_surface(code)
    file_surface_score = (float(_fuzz.partial_ratio(query_norm, file_norm))
                          if query_norm and file_norm else 0.0)

    module_imports = [
        ast.unparse(stmt)
        for stmt in tree.body
        if isinstance(stmt, (ast.Import, ast.ImportFrom))
    ]

    candidates = []

    for container in ast.walk(tree):
        body = getattr(container, 'body', None)
        if not isinstance(body, list):
            continue

        for start_idx, stmt in enumerate(body):
            var_name = _quantum_circuit_assignment(stmt)
            if var_name is None:
                continue

            def mutates_target(s, _v=var_name):
                for node in ast.walk(s):
                    if (isinstance(node, ast.Call)
                            and isinstance(node.func, ast.Attribute)
                            and isinstance(node.func.value, ast.Name)
                            and node.func.value.id == _v):
                        return True
                return False

            included = {start_idx}
            for idx in range(start_idx + 1, len(body)):
                next_var = _quantum_circuit_assignment(body[idx])
                if next_var == var_name:
                    break
                if mutates_target(body[idx]):
                    included.add(idx)
                elif (var_name in _assigned_names(body[idx])
                        and var_name in _names_loaded(body[idx])):
                    included.add(idx)

            max_idx = max(included)
            needed = set()
            for idx in included:
                needed |= _names_loaded(body[idx])

            changed = True
            while changed:
                changed = False
                for idx in range(max_idx + 1):
                    if idx in included:
                        continue
                    if _assigned_names(body[idx]) & needed:
                        included.add(idx)
                        needed |= _names_loaded(body[idx])
                        changed = True

            ordered = sorted(included)
            selected_stmts = [ast.unparse(body[idx]) for idx in ordered]
            snippet_parts = module_imports + selected_stmts
            snippet = '\n'.join(dict.fromkeys(snippet_parts)).strip()
            snippet_norm = _normalize_for_surface(snippet)
            snippet_score = (float(_fuzz.partial_ratio(query_norm, snippet_norm))
                             if query_norm and snippet_norm else 0.0)
            source_nodes = [body[idx] for idx in ordered]
            candidates.append({
                'corpus_snippet': snippet,
                'target_var': var_name,
                'snippet_surface_score': snippet_score,
                'source_start_line': min(n.lineno for n in source_nodes),
                'source_end_line': max(getattr(n, 'end_lineno', n.lineno) for n in source_nodes),
                'snippet_length': len(snippet_norm),
            })

    if not candidates:
        return {'snippet_status': 'no_circuit_found', 'candidate_count': 0,
                'file_surface_score_recomputed': file_surface_score,
                'corpus_snippet': None, 'target_var': None}

    best = max(candidates, key=lambda x: (x['snippet_surface_score'], -x['snippet_length']))
    snippet_hash = _hashlib.sha256(best['corpus_snippet'].encode('utf-8')).hexdigest()
    return {
        'snippet_status': 'ok',
        'snippet_selection_method': 'max_partial_ratio_among_circuit_candidates',
        'candidate_count': len(candidates),
        'file_surface_score_recomputed': file_surface_score,
        'snippet_surface_score': best['snippet_surface_score'],
        'surface_score_gap': file_surface_score - best['snippet_surface_score'],
        'target_var': best['target_var'],
        'source_start_line': best['source_start_line'],
        'source_end_line': best['source_end_line'],
        'snippet_sha256': snippet_hash,
        'corpus_snippet': best['corpus_snippet'],
    }


In [148]:
def find_best_surface_snippet(code: str, query_code: str) -> dict:
    """
    Locate the highest-scoring surface-matched region first.
    Circuit availability is labelled only after selection.
    QC slice for execution is added separately by build_corpus_snippet_records
    via find_minimal_circuit_snippet.
    """
    from rapidfuzz import fuzz
    import ast
    import hashlib
    import re

    query_norm = _normalize_for_surface(
        _strip_preamble_for_surface(query_code)
    )

    file_norm, source_map = _normalize_with_source_map(code)

    if not query_norm or not file_norm:
        return {
            "snippet_status": "empty_normalized_text",
            "corpus_snippet": None,
        }

    alignment = fuzz.partial_ratio_alignment(
        query_norm,
        file_norm,
    )

    normalized_start = alignment.dest_start
    normalized_end = alignment.dest_end

    raw_start = source_map[normalized_start]
    raw_end = source_map[normalized_end - 1] + 1

    line_start = code.rfind("\n", 0, raw_start) + 1
    line_end = code.find("\n", raw_end)

    if line_end == -1:
        line_end = len(code)

    snippet = code[line_start:line_end].strip()

    source_start_line = code.count("\n", 0, line_start) + 1
    source_end_line = code.count("\n", 0, line_end) + 1

    try:
        snippet_tree = ast.parse(snippet)
        python_parse_status = "ok"
    except SyntaxError:
        snippet_tree = None
        python_parse_status = "syntax_error"

    has_direct_qc_constructor = bool(
        re.search(
            r'\b\w+\s*=\s*QuantumCircuit\s*\(',
            snippet,
        )
    )

    snippet_score = float(
        fuzz.partial_ratio(
            query_norm,
            _normalize_for_surface(snippet),
        )
    )

    return {
        "snippet_status": "located",
        "snippet_selection_method":
            "best_surface_alignment_before_circuit_filtering",
        "file_surface_score_recomputed": float(alignment.score),
        "snippet_surface_score": snippet_score,
        "surface_score_gap":
            float(alignment.score) - snippet_score,
        "normalized_match_start": normalized_start,
        "normalized_match_end": normalized_end,
        "source_start_line": source_start_line,
        "source_end_line": source_end_line,
        "python_parse_status": python_parse_status,
        "has_direct_quantum_circuit_constructor":
            has_direct_qc_constructor,
        "snippet_sha256": hashlib.sha256(
            snippet.encode("utf-8")
        ).hexdigest(),
        "corpus_snippet": snippet,
    }

In [149]:
_SNIPPET_DIR = _snip_Path('dataset/corpus_snippets')
_SNIPPET_DIR.mkdir(parents=True, exist_ok=True)


def build_corpus_snippet_records(matches: list, label: str) -> list:
    records = []
    for match in matches:
        # Surface-aligned snippet — for text/AST metrics (s_gate_ast)
        selected = find_best_surface_snippet(
            code=match['corpus_code'],
            query_code=match['solution'],
        )
        # QC slice — best QuantumCircuit-building snippet for execution
        qc_result = find_minimal_circuit_snippet(
            code=match['corpus_code'],
            query_code=match['solution'],
        )
        qc_ok = qc_result.get('snippet_status') == 'ok'
        records.append({
            'task_id': match['task_id'],
            'corpus_repo': match.get('corpus_repo'),
            'corpus_path': match.get('corpus_path'),
            'retrieval_S': match.get('S'),
            'retrieval_surface_score': match.get('surface_score'),
            'retrieval_semantic_score': match.get('semantic_score'),
            'retrieval_gate_score': match.get('gate_score'),
            **selected,
            'corpus_snippet_qc_slice': qc_result.get('corpus_snippet') if qc_ok else None,
            'qc_slice_target_var':     qc_result.get('target_var')     if qc_ok else None,
            'qc_slice_status':         qc_result.get('snippet_status'),
        })
    out_path = _SNIPPET_DIR / f'{label.lower()}_corpus_snippets.json'
    with open(out_path, 'w', encoding='utf-8') as _f:
        _json_snip.dump(records, _f, indent=2, ensure_ascii=False)
    n_ok = sum(1 for r in records if r.get('corpus_snippet_qc_slice') is not None)
    print(f'[{label}] {len(records)} records ({n_ok} with qc_slice) -> {out_path}')
    return records

qhe_snippet_records = build_corpus_snippet_records(qhe_stack_matches, label='QHE')
qb_snippet_records  = build_corpus_snippet_records(qb_stack_matches,  label='QB')

qhe_snippet_lookup = {r['task_id']: r for r in qhe_snippet_records}
qb_snippet_lookup  = {r['task_id']: r for r in qb_snippet_records}
print(f'snippet lookups ready: QHE={len(qhe_snippet_lookup)}  QB={len(qb_snippet_lookup)}')

[QHE] 151 records (102 with qc_slice) -> dataset/corpus_snippets/qhe_corpus_snippets.json
[QB] 42 records (40 with qc_slice) -> dataset/corpus_snippets/qb_corpus_snippets.json
snippet lookups ready: QHE=151  QB=42


In [ ]:
# ---- corpus_code: locate + minimally execute, never run the whole file or call a function ----

def _names_loaded(stmt) -> set:
    return {n.id for n in ast.walk(stmt) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Load)}

def _strip_preamble_for_surface(code: str) -> str:
    """Strip everything before the first def/class to get the function body."""
    lines = code.splitlines()
    found_def = False
    body = []
    for line in lines:
        stripped = line.strip()
        if not found_def:
            if (stripped.startswith('def ') or stripped.startswith('async def ')
                    or stripped.startswith('class ')):
                found_def = True
            continue
        body.append(line)
    return '\n'.join(body) if body else code


def _bound_names(target) -> set:
    if isinstance(target, ast.Name):
        return {target.id}
    if isinstance(target, (ast.Tuple, ast.List)):
        result = set()
        for element in target.elts:
            result |= _bound_names(element)
        return result
    return set()


def _assigned_names(stmt) -> set:
    if isinstance(stmt, ast.Assign):
        result = set()
        for target in stmt.targets:
            result |= _bound_names(target)
        return result
    if isinstance(stmt, ast.AnnAssign):
        return _bound_names(stmt.target)
    if isinstance(stmt, ast.AugAssign):
        return _bound_names(stmt.target)
    if isinstance(stmt, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        return {stmt.name}
    if isinstance(stmt, ast.Import):
        return {alias.asname or alias.name.split('.')[0] for alias in stmt.names}
    if isinstance(stmt, ast.ImportFrom):
        return {alias.asname or alias.name for alias in stmt.names}
    return set()


def _quantum_circuit_assignment(stmt):
    """Return the assigned variable name if stmt constructs a QuantumCircuit, else None."""
    if isinstance(stmt, ast.Assign) and len(stmt.targets) == 1:
        target, value = stmt.targets[0], stmt.value
    elif isinstance(stmt, ast.AnnAssign):
        target, value = stmt.target, stmt.value
    else:
        return None
    if not isinstance(target, ast.Name) or not isinstance(value, ast.Call):
        return None
    func = value.func
    is_qc = (
        (isinstance(func, ast.Name) and func.id == 'QuantumCircuit')
        or (isinstance(func, ast.Attribute) and func.attr == 'QuantumCircuit')
    )
    return target.id if is_qc else None


In [180]:
# ═══════════════════════════════════════════════════════════════════════
# Section N: Spearman correlation — corpus similarity vs. pass@1 (funnel)
#
# Table 1: QHE then QB, metrics sorted by valid-n descending (funnel).
# Table 2: common-task subset (all 6 metrics non-None); QB n=24, QHE n=7.
#
# CI  : percentile bootstrap, 20 000 iterations, seed = 42 + j
# p   : two-sided permutation, 100 000 iterations, seed = 2042 + j
# Holm: step-down correction over 6 tests within each benchmark
# Ties: average-rank (scipy default)
# ═══════════════════════════════════════════════════════════════════════
import numpy as np
from scipy.stats import spearmanr

# ── helpers ──────────────────────────────────────────────────────────────

def _sp_r(x, y):
    r = spearmanr(x, y).statistic
    return float(r) if not np.isnan(r) else None

def _bootstrap_ci(x, y, n_boot=20_000, seed=42, ci=0.95):
    rng = np.random.default_rng(seed)
    n   = len(x)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        r = _sp_r(x[idx], y[idx])
        if r is not None:
            boots.append(r)
    if len(boots) < 100:
        return None, None
    lo = float(np.percentile(boots, 100 * (1 - ci) / 2))
    hi = float(np.percentile(boots, 100 * (1 + ci) / 2))
    return lo, hi

def _perm_pvalue(x, y, n_perm=100_000, seed=2042):
    rng  = np.random.default_rng(seed)
    obs  = _sp_r(x, y)
    if obs is None:
        return None
    obs  = abs(obs)
    count = sum(
        1 for _ in range(n_perm)
        if abs(_sp_r(x, rng.permutation(y)) or 0) >= obs
    )
    return (count + 1) / (n_perm + 1)

def _holm(pvals):
    m, order, adj, run = len(pvals), np.argsort(pvals), np.empty(len(pvals)), 0.0
    for rank, idx in enumerate(order):
        p = min(1.0, pvals[idx] * (m - rank))
        run = max(run, p)
        adj[idx] = run
    return adj

# ── core: compute one row per metric for given task subset ───────────────

METRICS = [
    ("retrieval_S",        "retrieval_S"),
    ("s_gate_ast",         "s_gate_ast"),
    ("s_gate_raw",         "s_gate_raw"),
    ("s_struct_raw_lcs",   "s_struct_lcs"),
    ("s_struct_raw_kgram", "s_struct_kgram"),
    ("f",                  "f"),
]

def _compute_rows(corpus, passk, task_ids=None):
    if task_ids is None:
        task_ids = [tid for tid in passk if tid in corpus]
    rows   = []
    raw_ps = []
    for j, (field, disp) in enumerate(METRICS):
        pairs = [
            (float(corpus[tid][field]), float(passk[tid]))
            for tid in task_ids
            if tid in corpus and tid in passk and corpus[tid].get(field) is not None
        ]
        n = len(pairs)
        if n < 4:
            rows.append(dict(j=j, disp=disp, n=n,
                             r=None, lo=None, hi=None,
                             p_raw=None, p_holm=None))
            raw_ps.append(None)
            continue
        x = np.array([p[0] for p in pairs])
        y = np.array([p[1] for p in pairs])
        r      = _sp_r(x, y)
        lo, hi = _bootstrap_ci(x, y, seed=42   + j)
        p_raw  = _perm_pvalue(x, y,  seed=2042 + j)
        rows.append(dict(j=j, disp=disp, n=n,
                         r=r, lo=lo, hi=hi,
                         p_raw=p_raw, p_holm=None))
        raw_ps.append(p_raw)
    # Holm correction (only over metrics with a valid p-value)
    valid_idx = [i for i, p in enumerate(raw_ps) if p is not None]
    if valid_idx:
        adj = _holm(np.array([raw_ps[i] for i in valid_idx]))
        for k, i in enumerate(valid_idx):
            rows[i]["p_holm"] = float(adj[k])
    return rows

# ── transposed display ────────────────────────────────────────────────────

def _fmt_sig(p):
    if p is None: return "—"
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "." if p < 0.10 else "ns"

def _print_transposed(rows_list, labels, title, sort_by_n=True):
    col_w = 17
    ref   = rows_list[0]
    order = (sorted(range(len(ref)), key=lambda i: -ref[i]["n"])
             if sort_by_n else list(range(len(ref))))

    sep = "─" * (16 + col_w * len(order))
    hdr = f"{'':14}" + "".join(f"{ref[i]['disp']:>{col_w}}" for i in order)

    print(f"\n{title}")
    print(sep)
    print(hdr)
    print(sep)

    STAT_ROWS = [
        ("n",       lambda r: str(r["n"])),
        ("r",       lambda r: f"{r['r']:+.4f}"                        if r["r"]     is not None else "—"),
        ("95% CI",  lambda r: f"[{r['lo']:+.3f},{r['hi']:+.3f}]"      if r["lo"]    is not None else "—"),
        ("p_raw",   lambda r: f"{r['p_raw']:.5f}"                     if r["p_raw"] is not None else "—"),
        ("p_holm",  lambda r: f"{r['p_holm']:.5f}"                    if r["p_holm"]is not None else "—"),
        ("sig",     lambda r: _fmt_sig(r["p_holm"])),
    ]

    for b_idx, (rows, blabel) in enumerate(zip(rows_list, labels)):
        if b_idx > 0:
            print(sep)
        print(f"  [{blabel}]")
        for stat_name, fmt in STAT_ROWS:
            line = f"  {stat_name:<12}" + "".join(f"{fmt(rows[i]):>{col_w}}" for i in order)
            print(line)
    print(sep)

# ── load data ────────────────────────────────────────────────────────────

_SQDIR = REPO_DIR / "dataset" / "s_quantum"

def _load_passk(exec_records):
    return {
        r["task_id"]: sum(r["exec_results"]) / len(r["exec_results"])
        for r in exec_records if r.get("exec_results")
    }

_qb_corpus  = {r["task_id"]: r for r in json.load(open(_SQDIR / "qb_corpus_metrics.json"))}
_qhe_corpus = {r["task_id"]: r for r in json.load(open(_SQDIR / "qhe_corpus_metrics.json"))}
_qb_passk   = _load_passk(qb_exec)
_qhe_passk  = _load_passk(qhe_exec)

# ── Table 1: full funnel (QHE first) ─────────────────────────────────────

print("Computing Table 1 (full set) ...")
_qhe_rows = _compute_rows(_qhe_corpus, _qhe_passk)
_qb_rows  = _compute_rows(_qb_corpus,  _qb_passk)

_print_transposed(
    [_qhe_rows, _qb_rows],
    ["QHE (151 tasks)", "QB  (42 tasks)"],
    "Table 1 — Spearman(metric, pass@1)  |  columns sorted by QHE valid-n (funnel)",
)

# ── Table 2: common-task subset (all 6 metrics non-None, QHE first) ──────

_all_fields = [f for f, _ in METRICS]

_qhe_common = sorted(
    tid for tid in _qhe_passk
    if tid in _qhe_corpus
    and all(_qhe_corpus[tid].get(f) is not None for f in _all_fields)
)
_qb_common = sorted(
    tid for tid in _qb_passk
    if tid in _qb_corpus
    and all(_qb_corpus[tid].get(f) is not None for f in _all_fields)
)

print(f"\nQHE common tasks (n={len(_qhe_common)}): {_qhe_common}")
print(f"QB  common tasks (n={len(_qb_common)}):  {_qb_common}")

print("\nComputing Table 2 (common-task subset) ...")
_qhe_com_rows = _compute_rows(_qhe_corpus, _qhe_passk, task_ids=_qhe_common)
_qb_com_rows  = _compute_rows(_qb_corpus,  _qb_passk,  task_ids=_qb_common)

_print_transposed(
    [_qhe_com_rows, _qb_com_rows],
    [f"QHE (common n={len(_qhe_common)})", f"QB  (common n={len(_qb_common)})"],
    "Table 2 — Spearman on common-task subset (all 6 metrics non-None)  |  METRICS order",
    sort_by_n=False,
)


Computing Table 1 (full set) ...


/var/folders/09/d7nq6hrs3bb2839jr3tllsvw0000gn/T/ipykernel_36843/766713722.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r = spearmanr(x, y).statistic



Table 1 — Spearman(metric, pass@1)  |  columns sorted by QHE valid-n (funnel)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                    retrieval_S       s_gate_ast       s_gate_raw     s_struct_lcs   s_struct_kgram                f
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  [QHE (151 tasks)]
  n                         151               47               39               39               39               11
  r                     +0.2423          +0.0044          -0.3586          -0.2606          -0.1579          +0.5556
  95% CI        [+0.088,+0.389]  [-0.299,+0.320]  [-0.636,-0.024]  [-0.540,+0.095]  [-0.434,+0.163]  [-0.029,+0.890]
  p_raw                 0.00288          0.97672          0.02589          0.10775          0.33517          0.08153
  p_holm                0.01728          0.97672          0.12945          0.3

In [14]:
# ═══════════════════════════════════════════════════════════════════════
# Section N (extended): Spearman + WL & edge-Jaccard structural metrics
#
# Same analysis as Section N above, but METRICS expanded with:
#   s_struct_wl_raw           (WL subtree kernel cosine similarity, [0,1])
#   s_struct_edge_jaccard_raw (Jaccard of qubit-interaction edge sets, [0,1])
#
# Self-contained: all helpers copied; data loaded from JSON (pass1_20).
# ═══════════════════════════════════════════════════════════════════════
import json as _json2, numpy as _np2
from pathlib import Path
from scipy.stats import spearmanr as _spearmanr2

# ── helpers (copied from Section N) ──────────────────────────────────────

def _sp_r2(x, y):
    r = _spearmanr2(x, y).statistic
    return float(r) if not _np2.isnan(r) else None

def _bootstrap_ci2(x, y, n_boot=20_000, seed=42, ci=0.95):
    rng = _np2.random.default_rng(seed)
    n   = len(x)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        r = _sp_r2(x[idx], y[idx])
        if r is not None:
            boots.append(r)
    if len(boots) < 100:
        return None, None
    lo = float(_np2.percentile(boots, 100 * (1 - ci) / 2))
    hi = float(_np2.percentile(boots, 100 * (1 + ci) / 2))
    return lo, hi

def _perm_pvalue2(x, y, n_perm=100_000, seed=2042):
    rng  = _np2.random.default_rng(seed)
    obs  = _sp_r2(x, y)
    if obs is None:
        return None
    obs  = abs(obs)
    count = sum(
        1 for _ in range(n_perm)
        if abs(_sp_r2(x, rng.permutation(y)) or 0) >= obs
    )
    return (count + 1) / (n_perm + 1)

def _holm2(pvals):
    m, order, adj, run = len(pvals), _np2.argsort(pvals), _np2.empty(len(pvals)), 0.0
    for rank, idx in enumerate(order):
        p = min(1.0, pvals[idx] * (m - rank))
        run = max(run, p)
        adj[idx] = run
    return adj

def _fmt_sig2(p):
    if p is None: return "—"
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "." if p < 0.10 else "ns"

def _print_transposed2(rows_list, labels, title, sort_by_n=True):
    col_w = 17
    ref   = rows_list[0]
    order = (sorted(range(len(ref)), key=lambda i: -ref[i]["n"])
             if sort_by_n else list(range(len(ref))))
    sep = "─" * (16 + col_w * len(order))
    hdr = f"{'':14}" + "".join(f"{ref[i]['disp']:>{col_w}}" for i in order)
    print(f"\n{title}")
    print(sep)
    print(hdr)
    print(sep)
    STAT_ROWS = [
        ("n",       lambda r: str(r["n"])),
        ("r",       lambda r: f"{r['r']:+.4f}"                        if r["r"]     is not None else "—"),
        ("95% CI",  lambda r: f"[{r['lo']:+.3f},{r['hi']:+.3f}]"      if r["lo"]    is not None else "—"),
        ("p_raw",   lambda r: f"{r['p_raw']:.5f}"                     if r["p_raw"] is not None else "—"),
        ("p_holm",  lambda r: f"{r['p_holm']:.5f}"                    if r["p_holm"]is not None else "—"),
        ("sig",     lambda r: _fmt_sig2(r["p_holm"])),
    ]
    for b_idx, (rows, blabel) in enumerate(zip(rows_list, labels)):
        if b_idx > 0:
            print(sep)
        print(f"  [{blabel}]")
        for stat_name, fmt in STAT_ROWS:
            line = f"  {stat_name:<12}" + "".join(f"{fmt(rows[i]):>{col_w}}" for i in order)
            print(line)
    print(sep)

# ── extended METRICS (8 metrics) ─────────────────────────────────────────
METRICS_EXT = [
    ("retrieval_S",               "retrieval_S"),
    ("s_gate_ast",                "s_gate_ast"),
    ("s_gate_raw",                "s_gate_raw"),
    ("s_struct_raw_lcs",          "s_struct_lcs"),
    ("s_struct_raw_kgram",        "s_struct_kgram"),
    ("s_struct_wl_raw",           "s_struct_wl"),
    ("s_struct_edge_jaccard_raw", "s_struct_ej"),
    ("f",                         "f"),
]

def _compute_rows_ext(corpus, passk, task_ids=None):
    if task_ids is None:
        task_ids = [tid for tid in passk if tid in corpus]
    rows   = []
    raw_ps = []
    for j, (field, disp) in enumerate(METRICS_EXT):
        pairs = [
            (float(corpus[tid][field]), float(passk[tid]))
            for tid in task_ids
            if tid in corpus and tid in passk and corpus[tid].get(field) is not None
        ]
        n = len(pairs)
        if n < 4:
            rows.append(dict(j=j, disp=disp, n=n,
                             r=None, lo=None, hi=None,
                             p_raw=None, p_holm=None))
            raw_ps.append(None)
            continue
        x = _np2.array([p[0] for p in pairs])
        y = _np2.array([p[1] for p in pairs])
        r      = _sp_r2(x, y)
        lo, hi = _bootstrap_ci2(x, y, seed=42   + j)
        p_raw  = _perm_pvalue2(x, y,  seed=2042 + j)
        rows.append(dict(j=j, disp=disp, n=n,
                         r=r, lo=lo, hi=hi,
                         p_raw=p_raw, p_holm=None))
        raw_ps.append(p_raw)
    valid_idx = [i for i, p in enumerate(raw_ps) if p is not None]
    if valid_idx:
        adj = _holm2(_np2.array([raw_ps[i] for i in valid_idx]))
        for k, i in enumerate(valid_idx):
            rows[i]["p_holm"] = float(adj[k])
    return rows

# ── load data ─────────────────────────────────────────────────────────────
_EXEC_DIR2 = Path(REPO_DIR) / "dataset"
_SQDIR2    = Path(REPO_DIR) / "dataset" / "s_quantum"

_qhe_passk2  = {r["task_id"]: r["pass1_20"]
                for r in _json2.load(open(_EXEC_DIR2 / "starcoder_qhe_exec.json"))
                if "pass1_20" in r}
_qb_passk2   = {r["task_id"]: r["pass1_20"]
                for r in _json2.load(open(_EXEC_DIR2 / "starcoder_qb_exec.json"))
                if "pass1_20" in r}
_qhe_corpus2 = {r["task_id"]: r
                for r in _json2.load(open(_SQDIR2 / "qhe_corpus_metrics.json"))}
_qb_corpus2  = {r["task_id"]: r
                for r in _json2.load(open(_SQDIR2 / "qb_corpus_metrics.json"))}

# ── Table 1: full funnel ──────────────────────────────────────────────────
print("Computing Table 1-ext (full set, 8 metrics) ...")
_qhe_rows2 = _compute_rows_ext(_qhe_corpus2, _qhe_passk2)
_qb_rows2  = _compute_rows_ext(_qb_corpus2,  _qb_passk2)

_print_transposed2(
    [_qhe_rows2, _qb_rows2],
    ["QHE (151 tasks)", "QB  (42 tasks)"],
    "Table 1-ext — Spearman(metric, pass@1)  |  +WL & edge-Jaccard  |  sorted by QHE valid-n",
)

# ── Table 2: common-task subset (all 8 metrics non-None) ─────────────────
_all_fields_ext = [f for f, _ in METRICS_EXT]

_qhe_common2 = sorted(
    tid for tid in _qhe_passk2
    if tid in _qhe_corpus2
    and all(_qhe_corpus2[tid].get(f) is not None for f in _all_fields_ext)
)
_qb_common2 = sorted(
    tid for tid in _qb_passk2
    if tid in _qb_corpus2
    and all(_qb_corpus2[tid].get(f) is not None for f in _all_fields_ext)
)

print(f"\nQHE common tasks (all 8 non-None, n={len(_qhe_common2)}): {_qhe_common2}")
print(f"QB  common tasks (all 8 non-None, n={len(_qb_common2)}):  {_qb_common2}")

print("\nComputing Table 2-ext (common-task subset) ...")
_qhe_com_rows2 = _compute_rows_ext(_qhe_corpus2, _qhe_passk2, task_ids=_qhe_common2)
_qb_com_rows2  = _compute_rows_ext(_qb_corpus2,  _qb_passk2,  task_ids=_qb_common2)

_print_transposed2(
    [_qhe_com_rows2, _qb_com_rows2],
    [f"QHE (common n={len(_qhe_common2)})", f"QB  (common n={len(_qb_common2)})"],
    "Table 2-ext — Spearman on common-task subset (all 8 metrics non-None)  |  METRICS order",
    sort_by_n=False,
)

Computing Table 1-ext (full set, 8 metrics) ...


/var/folders/09/d7nq6hrs3bb2839jr3tllsvw0000gn/T/ipykernel_99532/4279445703.py:17: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r = _spearmanr2(x, y).statistic



Table 1-ext — Spearman(metric, pass@1)  |  +WL & edge-Jaccard  |  sorted by QHE valid-n
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                    retrieval_S       s_gate_ast      s_struct_wl       s_gate_raw     s_struct_lcs   s_struct_kgram      s_struct_ej                f
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  [QHE (151 tasks)]
  n                         151               47               42               39               39               39               36               11
  r                     +0.2423          +0.0044          -0.0382          -0.3586          -0.2606          -0.1579          -0.0683          +0.5556
  95% CI        [+0.088,+0.389]  [-0.299,+0.320]  [-0.338,+0.276]  [-0.636,-0.024]  [-0.540,+0.095]  [-0.434,+0.163]  [-0.370,+0.258

# Section N+1: Global Moran's I
## W built from corpus metric, x = pass@1

For each metric, W ranks tasks by |score_i - score_j| (build_score_rank_weights).
Sections: K=5 QHE, K=10 QHE, K=5 QB, K=10 QB
Rows: n, p (p_sim), I (Global Moran's I), z (z_sim)
Cols: retrieval_S, s_gate_ast, s_gate_raw, s_struct_lcs, s_struct_kgram, f


In [8]:
# ══════════════════════════════════════════════════════════════════
#  Section 16A. 手动构造 W —— 按导师定义：rank |score_i - score_j|，取前 k 个
#  不用 KNN 模型、不用欧氏距离框架，纯排序
# ══════════════════════════════════════════════════════════════════
from esda.moran import Moran
from libpysal.weights import W as PysalW

def build_score_rank_weights(score_dict: dict, k_neighbors: int):
    """
    score_dict: {task_id: score}
    对每个task: dist(i,j) = |score_i - score_j|，升序排序，取前k_neighbors个(不含自己)作为邻居。
    返回 (neighbors, weights, ids) —— 喂给 libpysal.weights.W 的标准格式。
    按定义保留非对称(i是j的近邻 不代表 j是i的近邻)。
    """
    ids    = list(score_dict.keys())
    n      = len(ids)
    scores = np.array([score_dict[t] for t in ids], dtype=float)
    k_eff  = min(k_neighbors, n - 1)

    neighbors, weights = {}, {}
    for idx, tid in enumerate(ids):
        dist = np.abs(scores - scores[idx])#距离 = |score_i - score_j|
        dist[idx] = np.inf                       # 排除自己
        order = np.argsort(dist, kind='stable')  # 升序：最近的排前面
        nbr_idx = order[:k_eff]
        neighbors[tid] = [ids[j] for j in nbr_idx]
        weights[tid]   = [1.0] * k_eff
    return neighbors, weights, ids

print('build_score_rank_weights 定义完成')


build_score_rank_weights 定义完成


In [ ]:
import numpy as np
import pandas as pd
from esda.moran import Moran
from libpysal.weights import W as PysalW
import json as _json

K_VALUES    = [3, 4, 5, 6, 7, 10]
PERMUTATIONS = 9999
SEED        = 42

METRICS = [
    ("retrieval_S",               "retrieval_S"),
    ("s_gate_ast",                "s_gate_ast"),
    ("s_gate_raw",                "s_gate_raw"),
    ("s_struct_raw_lcs",          "s_struct_lcs"),
    ("s_struct_raw_kgram",        "s_struct_kgram"),
    ("s_struct_wl_raw",           "s_struct_wl"),
    ("s_struct_edge_jaccard_raw", "s_struct_ej"),
    ("f",                         "f"),
]

# ── load data (self-contained, no dependency on earlier cells) ────────────
_EXEC_DIR = REPO_DIR / "dataset"
_SQDIR    = REPO_DIR / "dataset" / "s_quantum"

_qhe_exec_raw = _json.load(open(_EXEC_DIR / "starcoder_qhe_exec.json"))
_qb_exec_raw  = _json.load(open(_EXEC_DIR / "starcoder_qb_exec.json"))
_qb_corpus    = {r["task_id"]: r for r in _json.load(open(_SQDIR / "qb_corpus_metrics.json"))}
_qhe_corpus   = {r["task_id"]: r for r in _json.load(open(_SQDIR / "qhe_corpus_metrics.json"))}

# ── Moran k-scan ─────────────────────────────────────────────────────────

def moran_k_scan(corpus, exec_raw, benchmark):
    records = {
        r["task_id"]: r
        for r in exec_raw
        if r.get("exec_results")
    }

    rows = []

    for field, label in METRICS:
        scores = {}
        for tid in records:
            value = corpus.get(tid, {}).get(field)
            if value is None:
                continue
            value = float(value)
            if np.isfinite(value):
                scores[tid] = value

        n = len(scores)

        for k in K_VALUES:
            row = {
                "benchmark": benchmark,
                "metric":    label,
                "n":         n,
                "k":         k,
                "I":         np.nan,
                "p_perm":    np.nan,
                "status":    "ok",
            }

            if not 1 <= k <= n - 2:
                row["status"] = "invalid_k"
                rows.append(row)
                continue

            if len(set(scores.values())) < 2:
                row["status"] = "constant_score"
                rows.append(row)
                continue

            neighbors, weights, ids = build_score_rank_weights(scores, k)
            w = PysalW(neighbors, weights, id_order=ids)# Build spatial weights matrix
            w.transform = "r" # raw standardization

            x = np.array(
                [records[tid]["pass1_20"] for tid in ids],
                dtype=float,
            )

            if not np.all(np.isfinite(x)) or np.var(x) == 0:
                row["status"] = "invalid_or_constant_pass1"
                rows.append(row)
                continue

            prev_state = np.random.get_state()
            try:
                np.random.seed(SEED)
                m = Moran(x, w, permutations=PERMUTATIONS)
            finally:
                np.random.set_state(prev_state)

            sim_I = np.asarray(m.sim)
            row["I"] = float(m.I)

            if np.std(sim_I) < 1e-14:
                row["status"] = "degenerate_null"
            else:
                row["p_perm"] = float(
                    (1 + np.count_nonzero(sim_I >= m.I - 1e-12))
                    / (PERMUTATIONS + 1)
                )

            rows.append(row)

    return pd.DataFrame(rows)


moran_scan = pd.concat([
    moran_k_scan(_qhe_corpus, _qhe_exec_raw, "QHE"),
    moran_k_scan(_qb_corpus,  _qb_exec_raw,  "QB"),
], ignore_index=True)

print(f"Permutations={PERMUTATIONS}, seed={SEED}")
print("p_perm: unadjusted upper-tail permutation p")
print(
    moran_scan.to_string(
        index=False,
        na_rep="—",
        formatters={
            "I":      lambda v: f"{v:+.4f}",
            "p_perm": lambda v: f"{v:.4f}",
        },
    )
)


/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  warnings.warn(message)
/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  warnings.warn(message)
/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  warnings.warn(message)
/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  warnings.warn(message)
/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 5 

Permutations=9999, seed=42
p_perm: unadjusted upper-tail permutation p
benchmark         metric   n  k       I p_perm    status
      QHE    retrieval_S 151  3 +0.0645 0.1301        ok
      QHE    retrieval_S 151  4 +0.0207 0.2981        ok
      QHE    retrieval_S 151  5 +0.0376 0.1793        ok
      QHE    retrieval_S 151  6 +0.0435 0.1293        ok
      QHE    retrieval_S 151  7 +0.0510 0.0880        ok
      QHE    retrieval_S 151 10 +0.0606 0.0386        ok
      QHE     s_gate_ast  47  3 +0.0031 0.3510        ok
      QHE     s_gate_ast  47  4 +0.0003 0.3459        ok
      QHE     s_gate_ast  47  5 -0.0733 0.7389        ok
      QHE     s_gate_ast  47  6 -0.0342 0.5128        ok
      QHE     s_gate_ast  47  7 -0.0266 0.4639        ok
      QHE     s_gate_ast  47 10 +0.0428 0.1172        ok
      QHE     s_gate_raw  39  3 +0.5062 0.0007        ok
      QHE     s_gate_raw  39  4 +0.4980 0.0002        ok
      QHE     s_gate_raw  39  5 +0.3783 0.0008        ok
      QHE     s_g

In [13]:
import numpy as np
import pandas as pd
from esda.moran import Moran
from libpysal.weights import W as PysalW
import json as _json

K_VALUES    = [3, 4, 5, 6, 7, 10]
PERMUTATIONS = 9999
SEED        = 42

METRICS = [
    ("retrieval_S",               "retrieval_S"),
    ("s_gate_ast",                "s_gate_ast"),
    ("s_gate_raw",                "s_gate_raw"),
    ("s_struct_raw_lcs",          "s_struct_lcs"),
    ("s_struct_raw_kgram",        "s_struct_kgram"),
    ("s_struct_wl_raw",           "s_struct_wl"),
    ("s_struct_edge_jaccard_raw", "s_struct_ej"),
    ("f",                         "f"),
]

# ── load data (self-contained, no dependency on earlier cells) ────────────
_EXEC_DIR = REPO_DIR / "dataset"
_SQDIR    = REPO_DIR / "dataset" / "s_quantum"

_qhe_exec_raw = _json.load(open(_EXEC_DIR / "starcoder_qhe_exec.json"))
_qb_exec_raw  = _json.load(open(_EXEC_DIR / "starcoder_qb_exec.json"))
_qb_corpus    = {r["task_id"]: r for r in _json.load(open(_SQDIR / "qb_corpus_metrics.json"))}
_qhe_corpus   = {r["task_id"]: r for r in _json.load(open(_SQDIR / "qhe_corpus_metrics.json"))}

# ── Moran k-scan ─────────────────────────────────────────────────────────

def moran_k_scan(corpus, exec_raw, benchmark):
    records = {
        r["task_id"]: r
        for r in exec_raw
        if r.get("exec_results")
    }

    rows = []

    for field, label in METRICS:
        scores = {}
        for tid in records:
            value = corpus.get(tid, {}).get(field)
            if value is None:
                continue
            value = float(value)
            if np.isfinite(value):
                scores[tid] = value

        n = len(scores)

        for k in K_VALUES:
            row = {
                "benchmark": benchmark,
                "metric":    label,
                "n":         n,
                "k":         k,
                "I":         np.nan,
                "p_perm":    np.nan,
                "status":    "ok",
            }

            if not 1 <= k <= n - 2:
                row["status"] = "invalid_k"
                rows.append(row)
                continue

            if len(set(scores.values())) < 2:
                row["status"] = "constant_score"
                rows.append(row)
                continue

            neighbors, weights, ids = build_score_rank_weights(scores, k)
            w = PysalW(neighbors, weights, id_order=ids)
            w.transform = "r"

            x = np.array(
                [records[tid]["pass1_20"] for tid in ids],
                dtype=float,
            )

            if not np.all(np.isfinite(x)) or np.var(x) == 0:
                row["status"] = "invalid_or_constant_pass1"
                rows.append(row)
                continue

            prev_state = np.random.get_state()
            try:
                np.random.seed(SEED)
                m = Moran(x, w, permutations=PERMUTATIONS)
            finally:
                np.random.set_state(prev_state)

            sim_I = np.asarray(m.sim)
            row["I"] = float(m.I)

            if np.std(sim_I) < 1e-14:
                row["status"] = "degenerate_null"
            else:
                row["p_perm"] = float(m.p_sim)

            rows.append(row)

    return pd.DataFrame(rows)


moran_scan = pd.concat([
    moran_k_scan(_qhe_corpus, _qhe_exec_raw, "QHE"),
    moran_k_scan(_qb_corpus,  _qb_exec_raw,  "QB"),
], ignore_index=True)

print(f"Permutations={PERMUTATIONS}, seed={SEED}")
print(
    moran_scan.to_string(
        index=False,
        na_rep="—",
        formatters={
            "I":      lambda v: f"{v:+.4f}",
            "p_perm": lambda v: f"{v:.4f}",
        },
    )
)


/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  warnings.warn(message)
/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  warnings.warn(message)
/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  warnings.warn(message)
/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  warnings.warn(message)
/Users/milly/Library/Python/3.9/lib/python/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 5 

Permutations=9999, seed=42
benchmark         metric   n  k       I p_perm    status
      QHE    retrieval_S 151  3 +0.0645 0.1301        ok
      QHE    retrieval_S 151  4 +0.0207 0.2981        ok
      QHE    retrieval_S 151  5 +0.0376 0.1793        ok
      QHE    retrieval_S 151  6 +0.0435 0.1293        ok
      QHE    retrieval_S 151  7 +0.0510 0.0880        ok
      QHE    retrieval_S 151 10 +0.0606 0.0386        ok
      QHE     s_gate_ast  47  3 +0.0031 0.3510        ok
      QHE     s_gate_ast  47  4 +0.0003 0.3459        ok
      QHE     s_gate_ast  47  5 -0.0733 0.2612        ok
      QHE     s_gate_ast  47  6 -0.0342 0.4873        ok
      QHE     s_gate_ast  47  7 -0.0266 0.4639        ok
      QHE     s_gate_ast  47 10 +0.0428 0.1172        ok
      QHE     s_gate_raw  39  3 +0.5062 0.0007        ok
      QHE     s_gate_raw  39  4 +0.4980 0.0002        ok
      QHE     s_gate_raw  39  5 +0.3783 0.0008        ok
      QHE     s_gate_raw  39  6 +0.4156 0.0001        ok
    